# 10. Создание нейросетей

Видео: [10. Создание нейросетей](https://www.youtube.com/watch?v=YiatESITMAg&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=11)

## О чём урок

Видео 9 дало **классы-сборщики** (`Sequential`, свой `nn.Module`, `ModuleList`/`Dict`).  
Видео 10 — **методы готовой модели** и сборка «модель + loss + Adam» под две задачи (как в проекте).

| метод / шаг | зачем |
|:---|:---|
| `model.state_dict()` | словарь весов по слоям → потом `save` / `load` (видео 13) |
| `model.parameters()` | то, что отдаём в `Adam(...)` |
| `model.train()` / `model.eval()` | режим обучения vs валидации (важно с Dropout/BatchNorm) |
| CE + Adam / MSE + Adam | классификация Fashion и регрессия квадратов |

Повтор классов из видео 9 здесь короткий: акцент на методах и «склейке» к оптимизатору.

> **Конспект:** голова под задачу — [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети.md) **§10**; чеклист — **§12**. Карта — [`0_как_учить_pytorch.md`](../0_как_учить_pytorch.md). Классы — тренажёр [`09/`](../09/09_классы_нейросетей.ipynb).

**Как смотреть.** ~11 мин — **кусками**:

| блок | о чём | где roughly |
|:---|:---|:---|
| 1 | `state_dict` | после повтора Sequential |
| 2 | `parameters` → Adam | там же |
| 3 | `train` / `eval` | там же |
| 4 | имена слоёв / `add_module` | середина |
| 5 | модели MNIST + квадраты + loss/Adam | конец |

**Как решать.** Замени `...` → проверка. Решения — `10_создание_нейросетей_решения.ipynb` после попытки.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)


def need(cond, msg):
    assert cond, msg


def need_tensor(x, name):
    assert isinstance(x, torch.Tensor), f"{name} — пока не тензор (замени ... на код)"


print("torch", torch.__version__)

## 1. <span style="color:#7CDBA0">state_dict</span> — снимок весов

> **Конспект:** веса слоёв — [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети.md) **§7**, **§12**.

`model.state_dict()` возвращает **словарь**: ключ → тензор параметра.

В `Sequential` без имён ключи — индексы слоёв: `0.weight`, `0.bias`, …  
**ReLU не попадает** в словарь: у него нет обучаемых весов.

Форма `Linear(784, 128).weight` — `(128, 784)` (out × in).

Позже (видео 13) именно `state_dict` сохраняют на диск.

In [ ]:
# 1.1 state_dict: ключи и форма весов
# Sequential: Linear(784,128) → ReLU → Linear(128,10)
torch.manual_seed(0)
net = ...

sd = ...          # словарь состояния
keys = ...        # list(sd.keys()) — какие ключи есть?
w0 = ...          # тензор весов первого Linear по ключу
print("keys:", keys)
print("w0 shape:", tuple(w0.shape) if isinstance(w0, torch.Tensor) else w0)

In [ ]:
need(isinstance(net, nn.Sequential), "net = nn.Sequential(...)")
need(isinstance(sd, dict), "sd = net.state_dict()")
need(keys == ["0.weight", "0.bias", "2.weight", "2.bias"], f"keys={keys} — у ReLU нет весов, индексы 0 и 2")
need_tensor(w0, "w0")
need(tuple(w0.shape) == (128, 784), f"w0.shape={tuple(w0.shape)}, ждали (128, 784)")
print("1.1 — верно")

## 2. <span style="color:#C9A0DC">parameters()</span> → оптимизатор

`model.parameters()` — генератор всех обучаемых тензоров (веса и bias).

Именно его передают в Adam:

```python
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
```

Число параметров: `sum(p.numel() for p in model.parameters())`.  
Для `784→128→10`: `784·128 + 128 + 128·10 + 10 = 101770`.

In [ ]:
# 2.1 parameters() → Adam + число параметров
# Та же архитектура 784→128→ReLU→10
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))

opt = ...                 # Adam(model.parameters(), lr=1e-3)
n_params = ...            # sum(p.numel() for p in model.parameters())
# руками: 784*128 + 128 + 128*10 + 10
n_manual = ...

print("n_params:", n_params, "| manual:", n_manual, "| opt:", type(opt).__name__)

In [ ]:
need(isinstance(opt, torch.optim.Adam), "opt = torch.optim.Adam(model.parameters(), lr=1e-3)")
need(n_params == 101770, f"n_params={n_params}, ждали 101770")
need(n_manual == n_params, f"n_manual={n_manual} должно совпасть с n_params")
print("2.1 — верно")

## 3. <span style="color:#E6C07B">train()</span> и <span style="color:#E6C07B">eval()</span>

| вызов | `model.training` | когда |
|:---|:---|:---|
| `model.train()` | `True` | перед шагами обучения |
| `model.eval()` | `False` | перед val / test / инференсом |

Сейчас без Dropout/BatchNorm разницы в числах почти нет, но **привычка обязательна**: с видео 15–16 без `eval()` метрики на val будут «шумными».

Где ставить в цикле — покажет видео 11.

In [ ]:
# 3.1 train() / eval() — флаг model.training
m = nn.Linear(4, 2)
was_train = ...       # True/False сразу после создания
m.eval()
after_eval = ...      # m.training после eval()
m.train()
after_train = ...     # m.training после train()

print("start:", was_train, "| after eval:", after_eval, "| after train:", after_train)

In [ ]:
need(was_train is True, "новая модель уже в train-режиме (training=True)")
need(after_eval is False, "после eval() → training=False")
need(after_train is True, "после train() → training=True")
print("3.1 — верно")

## 4. Имена слоёв через `add_module`

В видео 9 это уже было; здесь — коротко, как у автора в видео 10.

```python
net = nn.Sequential()
net.add_module("fc1", nn.Linear(784, 128))
```

Тогда: `net.fc1`, а в `state_dict` ключи `fc1.weight`, не `0.weight`.

In [ ]:
# 4.1 Имена слоёв: add_module + обращение по имени
# Sequential пустой → add_module('fc1', Linear(784,128)) → 'act' ReLU → 'fc2' Linear(128,10)
torch.manual_seed(0)
named = nn.Sequential()
...

layer_fc1 = ...   # named.fc1
out = named(torch.randn(4, 784))
print("fc1:", type(layer_fc1).__name__, "| out:", tuple(out.shape))
print("state keys:", list(named.state_dict().keys()))

In [ ]:
need(isinstance(named, nn.Sequential), "named = nn.Sequential()")
need([n for n, _ in named.named_children()] == ["fc1", "act", "fc2"], "имена через add_module")
need(isinstance(layer_fc1, nn.Linear), "layer_fc1 = named.fc1")
need_tensor(out, "out")
need(tuple(out.shape) == (4, 10), f"out.shape={tuple(out.shape)}")
need(list(named.state_dict().keys()) == ["fc1.weight", "fc1.bias", "fc2.weight", "fc2.bias"], "в state_dict имена fc1/fc2")
print("4.1 — верно")

## 5. Две задачи: один класс, разные голова / loss / Adam

Как в конце видео у автора: **один шаблон MLP**, два экземпляра.

| задача | вход после Flatten | выход | loss | opt |
|:---|:---|:---|:---|:---|
| классификация (Fashion/MNIST) | 784 | 10 | `CrossEntropyLoss` | `Adam(clf.parameters())` |
| регрессия (квадраты) | 4096 | 2 | `MSELoss` | `Adam(reg.parameters())` |

Картинку `(B,1,H,W)` сначала `nn.Flatten()` → вектор. Softmax в CE **не** вешаем на выход сети (как в уроке 8).

Обучение циклом — **видео 11**. Здесь только «собрали и проверили shape».

In [ ]:
# 5.1 Класс под обе задачи + loss + Adam (как в конце видео 10)
# Flatten → Linear(in_features, hidden) → ReLU → Linear(hidden, out_features)
# Fashion: картинка (B,1,28,28) → 10 классов; squares: (B,1,64,64) → 2 координаты

class TaskMLP(nn.Module):
    def __init__(self, in_features, out_features, hidden=128):
        super().__init__()
        ...

    def forward(self, x):
        ...


torch.manual_seed(0)
clf = ...   # TaskMLP(28*28, 10)  — классификация
reg = ...   # TaskMLP(64*64, 2)   — регрессия

# loss + оптимизатор (как у автора: CE / MSE + Adam lr=1e-3)
criterion_clf = ...
opt_clf = ...
criterion_reg = ...
opt_reg = ...

y_clf = clf(torch.randn(8, 1, 28, 28))
y_reg = reg(torch.randn(8, 1, 64, 64))
print(
    "clf:", tuple(y_clf.shape) if isinstance(y_clf, torch.Tensor) else y_clf,
    "| reg:", tuple(y_reg.shape) if isinstance(y_reg, torch.Tensor) else y_reg,
)
print("losses:", type(criterion_clf).__name__, type(criterion_reg).__name__)

In [ ]:
need(isinstance(clf, TaskMLP) and isinstance(reg, TaskMLP), "clf/reg = TaskMLP(...)")
need(isinstance(criterion_clf, nn.CrossEntropyLoss), "criterion_clf = nn.CrossEntropyLoss()")
need(isinstance(criterion_reg, nn.MSELoss), "criterion_reg = nn.MSELoss()")
need(isinstance(opt_clf, torch.optim.Adam) and isinstance(opt_reg, torch.optim.Adam), "Adam(...parameters()...)")
need_tensor(y_clf, "y_clf")
need_tensor(y_reg, "y_reg")
need(tuple(y_clf.shape) == (8, 10), f"clf out {tuple(y_clf.shape)}")
need(tuple(y_reg.shape) == (8, 2), f"reg out {tuple(y_reg.shape)}")
print("5.1 — верно")

## 6. Что мы сделали

| тема | зачем |
|:---|:---|
| `state_dict` | смотреть / потом сохранять веса |
| `parameters` + Adam | что обновляет градиентный спуск |
| `train` / `eval` | режимы до Dropout/BN и для цикла |
| один класс → clf + reg | как в проекте Fashion + квадраты |

Дальше: тренажёр закрыт → раздел **«Видео 10»** в `project/fashion_mnist.ipynb` (loss + Adam на уже готовых `FashionMLP` / `SquaresMLP`). Цикл эпох — видео 11.